<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M08/M08_Lab2_MultiAgent_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🏗️ M08 Lab 2 — Multi-Agent Tools & Processes</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐⭐ Difficulty: Advanced &nbsp;|&nbsp; ⏱ Time: ~20 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Build a more complex <b>4-agent content creation pipeline</b></li>
    <li>Understand <b>task dependencies</b> — how agent outputs feed into downstream tasks</li>
    <li>Compare <b>single-agent vs multi-agent</b> output quality</li>
    <li>Give agents <b>custom tools</b> using the <code>@tool</code> decorator</li>
    <li>Switch from <b>sequential</b> to <b>hierarchical</b> process with a manager LLM</li>
    <li>Apply <b>guardrails</b> (<code>max_iter</code>, <code>max_rpm</code>) to control agent behavior and costs</li>
  </ol>
</div>

## 📦 Setup

Before we start, let's install the required packages and set up our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai crewai crewai-tools
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
import os
from google.colab import userdata
from dads5250 import setup_openai, pp, show_info, compare_responses, quiz

client = setup_openai()
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ The Content Creation Pipeline</h2>
</div>

We'll build a **content production crew** that mirrors how a real content team works:

```
🔬 Topic Researcher  →  ✍️ Content Writer  →  🔍 SEO Optimizer  →  📝 Editor
    (finds facts)         (writes draft)        (adds SEO)          (polishes)
```

Each agent builds on the previous agent's output, creating a **pipeline** of increasingly refined content.

In [ ]:
# ============================================================
# 🧑 Define 4 Specialized Agents
# ============================================================
from crewai import Agent, Task, Crew, Process

TOPIC = "How AI is transforming supply chain management in 2025"

# Agent 1: Topic Researcher
topic_researcher = Agent(
    role="Topic Research Specialist",
    goal="Research the topic thoroughly and provide structured facts, statistics, and expert quotes",
    backstory="""You are an investigative researcher with a background in technology and 
    business journalism. You are meticulous about sourcing facts and always provide 
    specific data points, percentages, and examples rather than vague statements.""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

# Agent 2: Content Writer
content_writer = Agent(
    role="Senior Content Writer",
    goal="Transform research into an engaging, well-structured blog post",
    backstory="""You are a content strategist who has written for Harvard Business Review 
    and MIT Sloan Management Review. You write in a clear, authoritative style that 
    balances depth with readability. You use concrete examples and avoid jargon.""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

# Agent 3: SEO Optimizer
seo_optimizer = Agent(
    role="SEO & Content Optimization Specialist",
    goal="Optimize the content for search engines while maintaining readability and quality",
    backstory="""You are an SEO expert with deep knowledge of search algorithms and 
    content optimization. You know how to naturally incorporate keywords, write 
    compelling meta descriptions, and structure content for both readers and search engines. 
    You NEVER sacrifice readability for SEO.""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

# Agent 4: Editor
editor = Agent(
    role="Senior Editor",
    goal="Polish the final content for publication — grammar, flow, tone, and factual accuracy",
    backstory="""You are a former editor-in-chief with 20 years of experience. You have 
    an eagle eye for grammar, inconsistencies, and weak arguments. You ensure every 
    piece is publication-ready: clear, concise, compelling, and free of errors. 
    You also check that claims are well-supported and tone is consistent throughout.""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

print("✅ 4 agents created:")
for a in [topic_researcher, content_writer, seo_optimizer, editor]:
    print(f"   🧑 {a.role}")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Define Tasks with Dependencies</h2>
</div>

Each task builds on the previous one. The output of one agent becomes the input context for the next.

In [ ]:
# ============================================================
# 📋 Define Tasks (Sequential Pipeline)
# ============================================================

# Task 1: Research
research_task = Task(
    description=f"""Research the topic: '{TOPIC}'
    
    Provide:
    1. 5-7 key facts with specific statistics and data points
    2. 3 real company examples (with names and what they did)
    3. 2-3 expert perspectives or industry trends
    4. Common challenges and solutions
    5. Future outlook (next 2-3 years)
    
    Format as a structured research brief with clear sections.""",
    expected_output="A structured research brief with facts, statistics, examples, and trends.",
    agent=topic_researcher
)

# Task 2: Write
writing_task = Task(
    description=f"""Using the research provided, write a 600-800 word blog post about: '{TOPIC}'
    
    Requirements:
    - Engaging hook in the first paragraph
    - 4-5 main sections with descriptive subheadings
    - Include the statistics and examples from the research
    - Clear conclusion with a forward-looking statement
    - Professional but accessible tone (HBR style)
    - Use bullet points and short paragraphs for readability""",
    expected_output="A well-written 600-800 word blog post with sections, examples, and data.",
    agent=content_writer
)

# Task 3: SEO Optimize
seo_task = Task(
    description=f"""Optimize the blog post for SEO:
    
    1. Suggest a compelling title (under 60 characters) with the primary keyword
    2. Write a meta description (150-160 characters)
    3. Identify 5 target keywords and ensure they appear naturally in the content
    4. Add internal linking suggestions (placeholder links)
    5. Optimize subheadings to include secondary keywords
    6. Ensure the content has proper heading hierarchy (H1, H2, H3)
    
    Return the FULL optimized blog post with SEO metadata at the top.
    Do NOT sacrifice readability for keyword stuffing.""",
    expected_output="The full optimized blog post with SEO metadata, keywords, and meta description.",
    agent=seo_optimizer
)

# Task 4: Final Edit
editing_task = Task(
    description="""Perform a final editorial review of the blog post:
    
    1. Fix any grammar, spelling, or punctuation errors
    2. Improve sentence flow and transitions between sections
    3. Ensure consistent tone throughout
    4. Verify all claims are supported by the data mentioned
    5. Tighten any wordy sections — every sentence should earn its place
    6. Ensure the opening hook is compelling and the conclusion is strong
    
    Return the FINAL publication-ready version with the SEO metadata intact.""",
    expected_output="The final, polished, publication-ready blog post with SEO metadata.",
    agent=editor
)

print("✅ 4 tasks defined — pipeline ready!")
print("   🔬 Research → ✍️ Write → 🔍 SEO → 📝 Edit")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Run the Content Crew</h2>
</div>

In [ ]:
# ============================================================
# 👥 Run the Content Creation Crew
# ============================================================

content_crew = Crew(
    agents=[topic_researcher, content_writer, seo_optimizer, editor],
    tasks=[research_task, writing_task, seo_task, editing_task],
    process=Process.sequential,
    verbose=True
)

print(f"🚀 Running content pipeline for: '{TOPIC}'")
print(f"   4 agents will work sequentially (~60 seconds)...\n")

multi_agent_result = content_crew.kickoff()

print("\n" + "="*70)
print("✅ PIPELINE COMPLETE")
print("="*70)

In [ ]:
# ============================================================
# 📄 Display Final Blog Post
# ============================================================
from IPython.display import display, Markdown

display(Markdown(str(multi_agent_result)))

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Single-Agent vs Multi-Agent Comparison</h2>
</div>

Let's see the difference. We'll ask a **single agent** to do everything the 4-agent crew just did.

In [ ]:
# ============================================================
# ⚔️ Single Agent: Same Task, One Agent
# ============================================================
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7)

single_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a content creator. Write an SEO-optimized, publication-ready 
    blog post. Include research, statistics, examples, SEO metadata, and polish it."""),
    ("user", f"Write a complete blog post about: '{TOPIC}'. Include SEO metadata, statistics, company examples, and make it publication-ready. 600-800 words.")
])

single_chain = single_prompt | llm | StrOutputParser()

print("🤖 Running single-agent version...\n")
single_agent_result = single_chain.invoke({})

print("✅ Single-agent result ready!")

In [ ]:
# ============================================================
# 📊 Side-by-Side Quality Comparison
# ============================================================

multi_text = str(multi_agent_result)
single_text = single_agent_result

# Visual side-by-side using compare_responses
compare_responses({
    "Single Agent (1 LLM call)": single_text[:500] + "..." if len(single_text) > 500 else single_text,
    "Multi-Agent (4 agents)": multi_text[:500] + "..." if len(multi_text) > 500 else multi_text,
})

# Metrics comparison
pp({
    "single_agent": {
        "word_count": len(single_text.split()),
        "has_seo_metadata": "meta" in single_text.lower(),
        "has_statistics": "%" in single_text,
        "agents_used": 1,
    },
    "multi_agent": {
        "word_count": len(multi_text.split()),
        "has_seo_metadata": "meta" in multi_text.lower(),
        "has_statistics": "%" in multi_text,
        "agents_used": 4,
    }
}, "Quality Comparison Metrics")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Multi-agent systems consistently produce higher quality output because each agent <b>specializes</b>. The researcher finds better facts, the writer crafts better prose, the SEO expert adds real optimization, and the editor catches errors. A single agent trying to do all four roles at once inevitably cuts corners.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ Adding Tools to Your Crew</h2>
</div>

In V5 we learned that agents can use **tools** — functions they call during execution to gather information or perform actions. CrewAI supports custom tools via the `@tool` decorator.

Let's give our **Topic Researcher** a real tool: a word counter that verifies article length requirements.

In [ ]:
# ============================================================
# 🔧 Create a Custom Tool with @tool
# ============================================================
from crewai.tools import tool

@tool("Word Counter")
def word_count_tool(text: str) -> str:
    """Count the number of words in a text. Use this to verify article length requirements."""
    count = len(text.split())
    return f"Word count: {count}"

@tool("Reading Time Estimator")
def reading_time_tool(text: str) -> str:
    """Estimate reading time for a text. Average adult reads 238 words per minute."""
    words = len(text.split())
    minutes = round(words / 238, 1)
    return f"Estimated reading time: {minutes} minutes ({words} words)"

# Test the tools directly
sample = "AI is transforming supply chains through demand forecasting and automation."
print(word_count_tool.run(sample))
print(reading_time_tool.run(sample))
print(f"\n✅ 2 custom tools created: {word_count_tool.name}, {reading_time_tool.name}")

In [ ]:
# ============================================================
# 🔧 Assign Tools to an Agent & Re-run the Crew
# ============================================================

# Give the editor the word count + reading time tools
editor_with_tools = Agent(
    role="Senior Editor",
    goal="Polish the final content for publication — check word count, grammar, flow, tone, and factual accuracy",
    backstory="""You are a former editor-in-chief with 20 years of experience. You have 
    an eagle eye for grammar, inconsistencies, and weak arguments. You always verify 
    the word count meets the target (600-800 words) using your tools before approving.""",
    verbose=True,
    allow_delegation=False,
    tools=[word_count_tool, reading_time_tool],  # <-- tools assigned here!
    llm="gpt-4.1-mini"
)

# Re-create the final editing task with the tool-equipped editor
editing_task_v2 = Task(
    description=f"""Perform a final editorial review of the blog post about '{TOPIC}'.
    
    1. Use the Word Counter tool to verify the article is 600-800 words
    2. Use the Reading Time Estimator to confirm reading time is under 4 minutes
    3. Fix any grammar, spelling, or punctuation errors
    4. Tighten wordy sections — every sentence should earn its place
    5. Report the final word count and reading time at the end
    
    Return the FINAL publication-ready version.""",
    expected_output="The final polished blog post with word count and reading time verification.",
    agent=editor_with_tools
)

# Build a new crew with the tool-equipped editor
tool_crew = Crew(
    agents=[topic_researcher, content_writer, seo_optimizer, editor_with_tools],
    tasks=[research_task, writing_task, seo_task, editing_task_v2],
    process=Process.sequential,
    verbose=True
)

print("🚀 Running crew WITH tools (editor can now count words)...")
print("   Watch the verbose output — you'll see the editor CALL the tool.\n")

tool_result = tool_crew.kickoff()

print("\n" + "="*70)
print("✅ TOOL-EQUIPPED PIPELINE COMPLETE")
print("="*70)

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Observation:</b> In the verbose output above, you should see the editor agent <b>calling</b> the Word Counter and Reading Time Estimator tools before finalizing. This is the key pattern: tools let agents <b>verify facts</b> and <b>gather data</b> during execution rather than guessing. The <code>@tool</code> decorator is all you need — define a Python function, add a docstring, and assign it via <code>tools=[...]</code>.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">6️⃣ Hierarchical Process</h2>
</div>

So far we have used `Process.sequential` — agents run in a fixed order. CrewAI also supports `Process.hierarchical`, where a **manager LLM** decides which agent to call next, can re-assign work, and coordinates the team dynamically.

Think of it like the difference between an assembly line (sequential) and a project manager running a meeting (hierarchical).

In [ ]:
# ============================================================
# 🏢 Hierarchical Process — Manager Delegates to Agents
# ============================================================

# Same 4 agents, same 4 tasks — but now a MANAGER coordinates them
hierarchical_crew = Crew(
    agents=[topic_researcher, content_writer, seo_optimizer, editor_with_tools],
    tasks=[research_task, writing_task, seo_task, editing_task_v2],
    process=Process.hierarchical,       # <-- the key change!
    manager_llm="gpt-4.1-mini",         # <-- manager model
    verbose=True
)

print("🏢 Running HIERARCHICAL crew (manager delegates tasks)...")
print("   Watch the manager decide which agent to call and when.\n")

hierarchical_result = hierarchical_crew.kickoff()

print("\n" + "="*70)
print("✅ HIERARCHICAL PIPELINE COMPLETE")
print("="*70)

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Observation:</b> Notice how the <b>manager</b> decides which agent to delegate each task to. In the verbose output you will see lines like <i>"I need to delegate this task to the Topic Research Specialist"</i>. The manager may also ask an agent to <b>redo</b> work if the output is not satisfactory. This is powerful for complex workflows where the optimal order is not obvious upfront.
</div>

**When to use each process:**

| Process | Best for | Trade-off |
|---------|----------|-----------|
| `sequential` | Well-defined pipelines with a clear order | Predictable, lower token cost |
| `hierarchical` | Complex tasks where delegation order is uncertain | More flexible, but uses extra tokens for the manager |

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">7️⃣ Guardrails: Controlling Agent Behavior</h2>
</div>

In production, you need guardrails to prevent agents from:
- **Looping forever** (an agent keeps retrying and never finishes)
- **Burning through API credits** (too many requests per minute)

CrewAI provides two key parameters for this.

In [ ]:
# ============================================================
# 🛡️ Guardrails — max_iter and max_rpm
# ============================================================

# max_iter: Maximum iterations an agent can take before being forced to stop.
# This prevents infinite loops when an agent keeps retrying a tool or rephrasing.
guardrailed_agent = Agent(
    role="Research Analyst",
    goal="Provide a brief summary of AI trends",
    backstory="You are a research analyst who works efficiently.",
    verbose=True,
    allow_delegation=False,
    max_iter=5,          # <-- Stop after 5 iterations (default is 25)
    llm="gpt-4.1-mini"
)

# max_rpm: Maximum requests per minute for the entire crew.
# This prevents hitting API rate limits and controls costs.
guardrailed_crew = Crew(
    agents=[guardrailed_agent],
    tasks=[Task(
        description="List 3 key AI trends for 2025 in one paragraph.",
        expected_output="A concise paragraph with 3 AI trends.",
        agent=guardrailed_agent
    )],
    process=Process.sequential,
    max_rpm=10,          # <-- Max 10 API calls per minute across all agents
    verbose=True
)

print("✅ Guardrails configured:")
print(f"   Agent max_iter = {guardrailed_agent.max_iter} (stops after 5 iterations)")
print(f"   Crew  max_rpm  = {guardrailed_crew.max_rpm} (max 10 requests/minute)")
print()

pp({
    "max_iter": {
        "where": "Agent(..., max_iter=5)",
        "why": "Prevents infinite loops — agent stops after N iterations",
        "default": 25,
        "recommendation": "5-10 for simple tasks, 15-25 for complex tool-using agents"
    },
    "max_rpm": {
        "where": "Crew(..., max_rpm=10)",
        "why": "Controls API costs and prevents rate limiting",
        "default": "None (unlimited)",
        "recommendation": "10-30 for development, higher for production"
    }
}, "Guardrail Parameters Reference")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What is the main advantage of multi-agent systems over single-agent for complex tasks?",
        "options": [
            "They use less API tokens",
            "Each agent specializes in one subtask, producing higher quality output through division of labor",
            "They run faster because agents work in parallel",
            "They don't require API keys"
        ],
        "answer": 1,
        "explanation": "Multi-agent systems leverage specialization — each agent focuses on what it does best (research, writing, editing), producing higher quality than one agent trying to do everything. This mirrors how human teams work."
    },
    {
        "q": "In a sequential CrewAI process, how does information flow between agents?",
        "options": [
            "All agents receive the same input simultaneously",
            "Each agent's output is automatically passed as context to the next agent's task",
            "A manager agent manually copies text between agents",
            "Agents share a common database"
        ],
        "answer": 1,
        "explanation": "In sequential mode, each task's output becomes part of the context for the next task. The writer sees the researcher's brief, the SEO expert sees the writer's draft, and the editor sees the optimized version."
    },
    {
        "q": "What does the @tool decorator do in CrewAI?",
        "options": [
            "It installs a new Python package",
            "It turns a Python function into a tool that agents can call during execution",
            "It creates a new agent automatically",
            "It connects to an external API"
        ],
        "answer": 1,
        "explanation": "The @tool decorator from crewai.tools wraps a regular Python function so that agents can discover it (via the docstring) and call it during their reasoning loop. You assign tools to agents via tools=[my_tool]."
    },
    {
        "q": "What is the key difference between Process.sequential and Process.hierarchical?",
        "options": [
            "Sequential is faster, hierarchical is slower",
            "Sequential runs agents in a fixed order; hierarchical uses a manager LLM to decide delegation dynamically",
            "Hierarchical does not support tools",
            "Sequential requires more agents than hierarchical"
        ],
        "answer": 1,
        "explanation": "In sequential mode, agents run in the exact order you list them. In hierarchical mode, a manager LLM coordinates the team — deciding which agent to call, when to re-delegate, and how to combine results."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Build Your Own Crew (with Tools!)

Design a crew for a domain of your choice. **Requirements:**
- At least **3 agents** with distinct roles
- At least **1 custom tool** (using the `@tool` decorator) assigned to one of your agents
- Choose either `Process.sequential` or `Process.hierarchical` and explain your choice

**Domain ideas:**
- **Marketing crew**: Market Researcher (with a trend-checker tool) → Copywriter → Campaign Manager
- **Code review crew**: Code Analyzer (with a line-counter tool) → Security Auditor → Documentation Writer
- **Hiring crew**: Resume Screener (with a keyword-matcher tool) → Interview Question Writer → Rubric Creator

In [ ]:
# ============================================================
# 🔧 Exercise: Build Your Own Crew (with Tools!)
# ============================================================
# Replace each ----- with the correct value

# Step 1: Create at least one custom tool
@tool("-----")  # Give your tool a name
def my_custom_tool(text: str) -> str:
    """-----"""  # Describe what it does (agents read this!)
    # YOUR CODE HERE
    return "-----"

# Step 2: Define your agents (at least 3, one with tools=[my_custom_tool])
agent_1 = Agent(
    role="-----",
    goal="-----",
    backstory="""-----""",
    verbose=True,
    allow_delegation=False,
    tools=[my_custom_tool],  # <-- at least one agent needs a tool
    llm="gpt-4.1-mini"
)

agent_2 = Agent(
    role="-----",
    goal="-----",
    backstory="""-----""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

agent_3 = Agent(
    role="-----",
    goal="-----",
    backstory="""-----""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

# Step 3: Define tasks
task_1 = Task(
    description="-----",
    expected_output="-----",
    agent=agent_1
)

task_2 = Task(
    description="-----",
    expected_output="-----",
    agent=agent_2
)

task_3 = Task(
    description="-----",
    expected_output="-----",
    agent=agent_3
)

# Step 4: Create your crew — pick sequential OR hierarchical
my_crew = Crew(
    agents=[agent_1, agent_2, agent_3],
    tasks=[task_1, task_2, task_3],
    process=Process.sequential,  # <-- or Process.hierarchical with manager_llm="gpt-4.1-mini"
    verbose=True
)

print("✅ Your custom crew is ready!")
print("   Uncomment the line below to run it.")

# my_result = my_crew.kickoff()
# display(Markdown(str(my_result)))

**📝 Your Observations:** *(double-click to edit)*

1. Compare the single-agent and multi-agent outputs. What specific improvements do you see in the multi-agent version? _[Your answer]_

2. When the editor used the Word Counter tool, did the final article meet the 600-800 word target? How does this compare to the version without tools? _[Your answer]_

3. What differences did you notice between the sequential and hierarchical crews? Which produced better output and why? _[Your answer]_

4. What domain did you choose for your custom crew? Why did you pick those specific roles and what tool did you create? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions:</p>
  <ol style="font-size: 14px;">
    <li>Explore <b>pre-built tools</b> from <code>crewai-tools</code> (e.g., <code>SerperDevTool</code> for web search, <code>FileReadTool</code> for reading files)</li>
    <li>Try enabling <b>memory</b> on a crew: <code>Crew(..., memory=True)</code> — agents remember context across tasks</li>
    <li>Experiment with <b>delegation</b>: set <code>allow_delegation=True</code> on agents so they can ask other agents for help</li>
    <li>Build a crew for your <b>hackathon project</b> or a real work task you face</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've built multi-agent crews with tools, hierarchical processes, and guardrails.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>Task dependencies</b> create a pipeline where each agent builds on the previous output</li>
      <li><b>Custom tools</b> via <code>@tool</code> let agents verify facts and gather data during execution</li>
      <li><b>Sequential vs hierarchical</b> — fixed pipeline vs dynamic manager delegation</li>
      <li><b>Guardrails</b> (<code>max_iter</code>, <code>max_rpm</code>) prevent runaway loops and control API costs</li>
      <li><b>Multi-agent</b> consistently beats single-agent for complex, multi-faceted tasks</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M09 Lab 1: Vision &amp; Evaluation</p>
</div>